In [28]:
import torch
from torch.utils import data
from collections import Counter
from torch import nn
epoch=300
batch_size = 64
num_steps = 20
lr=0.1

data_path = "../data/fra-eng/fra.txt"
device=torch.device('cuda')

In [29]:
def data_read():
    with open(data_path,'r') as file:
        text=file.read()
    return text

In [30]:
def preprocess_nmt(text):
    text = (
        text
        .replace("\u202f", " ")
        .replace("\xa0", " ")
        .lower()
    )
    result=[]
    for i,char in enumerate(text):
        if (
            char in ",.!?"
            and i > 0
            and text[i - 1] != " "
        ):
            result.append(' ')
        result.append(char)
    return "".join(result)

In [31]:
def tokenize_nmt(text,max_text=600):
    source=[]
    target=[]
    for line in text.splitlines():
        parts=line.split('\t')

        if len(parts)<2:
            continue
        source.append(parts[0].split())
        target.append(parts[1].split())
        if(max_text is not None and len(source)>=max_text):
            break
    return source,target

In [32]:
def vocab_maker(tokens,min_freq=2):
    counter=Counter()
    for line in tokens:
        counter.update(line)
    token_freq=sorted(counter.items(),key=lambda x:(-x[1],x[0]))
    idx_to_token=["<unk>",
        "<pad>",
        "<bos>",
        "<eos>"]
    for token,freq in token_freq:
        if freq<min_freq :
            continue
        if token not in idx_to_token:
            idx_to_token.append(token)
    token_to_idx={
        token:idx for idx,token in enumerate(idx_to_token)
    }
    vocab = {
        "idx_to_token": idx_to_token,
        "token_to_idx": token_to_idx
    }

    return vocab

In [33]:
def truncate_pad(line,num_step,pad_idx):
    if len(line)>num_step:
        return line[:num_step]
    return line+[pad_idx]*(num_step-len(line))


In [34]:
def build_array_nmt(lines, vocab, num_steps):
    token_to_idx = vocab["token_to_idx"]

    unk_idx = token_to_idx["<unk>"]
    pad_idx = token_to_idx["<pad>"]
    eos_idx = token_to_idx["<eos>"]

    array = []

    for line in lines:
        line_idx = [
            token_to_idx.get(token, unk_idx)
            for token in line
        ]

        # 每句话末尾添加结束标记
        line_idx.append(eos_idx)

        line_idx = truncate_pad(
            line_idx,
            num_steps,
            pad_idx
        )

        array.append(line_idx)

    array = torch.tensor(
        array,
        dtype=torch.long
    )

    valid_len = (
        array != pad_idx
    ).sum(dim=1)

    return array, valid_len

In [35]:
def load_data_nmt(
    batch_size,
    num_steps,
    num_examples=600
):
    raw_text = data_read()

    text = preprocess_nmt(raw_text)

    source, target = tokenize_nmt(
        text,
        num_examples
    )

    src_vocab = vocab_maker(
        source,
        min_freq=2
    )

    tgt_vocab = vocab_maker(
        target,
        min_freq=2
    )

    src_array, src_valid_len = build_array_nmt(
        source,
        src_vocab,
        num_steps
    )

    tgt_array, tgt_valid_len = build_array_nmt(
        target,
        tgt_vocab,
        num_steps
    )

    dataset = data.TensorDataset(
        src_array,
        src_valid_len,
        tgt_array,
        tgt_valid_len
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    return train_iter, src_vocab, tgt_vocab

In [36]:
train_iter,src_vocab,tgt_vocab=load_data_nmt(batch_size,num_steps)


In [37]:
class encoder(nn.Module):
    def __init__(self,input_num,hidden_layer,embedding_dim,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.hidden_layer=hidden_layer
        self.embedding_dim=embedding_dim
        self.embedding=nn.Embedding(input_num,embedding_dim,padding_idx=pad_idx)
        self.gru=nn.GRU(embedding_dim,hidden_layer,bidirectional=True)
    def forward(self,x):
        x=self.embedding(x)
        x=x.permute(1,0,2)
        output,state=self.gru(x)
        return state
    


In [38]:
class decoder(nn.Module):
    def __init__(self,input_num,hidden_layer,embedding_dim,pad_idx):
        super().__init__()
        self.input_num=input_num
        self.hidden_layer=hidden_layer
        self.embedding_dim=embedding_dim
        self.embedding=nn.Embedding(input_num,embedding_dim,padding_idx=pad_idx)
        self.gru=nn.GRU(embedding_dim+hidden_layer,hidden_layer)
        self.linear=nn.Linear(hidden_layer,input_num)
    def forward(self,x,state,new_info):
        x=self.embedding(x)
        x=x.permute(1,0,2) #step,batch,embedding
        x=torch.cat((x,new_info),dim=2)
        output,state=self.gru(x,state)
        output=output.reshape(-1,self.hidden_layer)
        output=self.linear(output)
        return output,state



In [39]:
class seq2seq(nn.Module):
    def __init__(self,encoder,decoder,bos_idx):
        super().__init__()
        self.encoder=encoder
        self.decoder=decoder
        self.linear=nn.Linear(encoder.hidden_layer*2,decoder.hidden_layer)
        self.bos_idx=bos_idx
    def forward(self,x,y):
        state=self.encoder(x)
        state1=state[0]
        state2=state[1]
        state=torch.cat((state1,state2),dim=1)
        state=self.linear(state)
        state=torch.tanh(state) #batch,hidden
        new_info=state.unsqueeze(0).repeat(num_steps,1,1) 
        decoder_state=state.unsqueeze(0).repeat(self.decoder.gru.num_layers,1,1) 
        bos=torch.full(size=(y.shape[0],1),fill_value=self.bos_idx,dtype=torch.long,device=y.device)
        y=torch.cat((bos,y[:,:-1]),dim=1)
        output,state=self.decoder(y,decoder_state,new_info)
        return output,state


In [40]:
src_pad_idx = src_vocab["token_to_idx"]["<pad>"]
tgt_pad_idx = tgt_vocab["token_to_idx"]["<pad>"]
bos_idx = tgt_vocab["token_to_idx"]["<bos>"]
my_encoder=encoder(len(src_vocab['token_to_idx']),256,128,src_pad_idx).to(device)
my_decoder=decoder(len(tgt_vocab['token_to_idx']),512,256,tgt_pad_idx).to(device)
net = seq2seq(
    my_encoder,
    my_decoder,
    bos_idx
).to(device)
trainer=torch.optim.SGD(net.parameters(),lr=lr)
loss = nn.CrossEntropyLoss(
    ignore_index=tgt_pad_idx
)


In [41]:
for i in range(epoch):
    net.train()
    train_loss=0
    train_count=0
    for x,x_len,y,y_len in train_iter:
        trainer.zero_grad()
        x=x.to(device)
        y=y.to(device)
        logit,state=net(x,y)
        y=y.T.reshape(-1)
        l=loss(logit,y)
        l.backward()
        torch.nn.utils.clip_grad_norm_(
            net.parameters(),
            max_norm=1.0
        )
        trainer.step()
        train_loss+=l.item()*y_len.sum().item()
        train_count+=y_len.sum().item()
    print(
        f"第{i + 1}个epoch，"
        f"loss：{train_loss / train_count:.4f}"
    )

第1个epoch，loss：4.7294
第2个epoch，loss：3.4182
第3个epoch，loss：2.8756
第4个epoch，loss：2.6903
第5个epoch，loss：2.5769
第6个epoch，loss：2.4927
第7个epoch，loss：2.4283
第8个epoch，loss：2.3753
第9个epoch，loss：2.3276
第10个epoch，loss：2.2842
第11个epoch，loss：2.2453
第12个epoch，loss：2.2087
第13个epoch，loss：2.1750
第14个epoch，loss：2.1459
第15个epoch，loss：2.1163
第16个epoch，loss：2.0886
第17个epoch，loss：2.0609
第18个epoch，loss：2.0375
第19个epoch，loss：2.0111
第20个epoch，loss：1.9872
第21个epoch，loss：1.9649
第22个epoch，loss：1.9409
第23个epoch，loss：1.9183
第24个epoch，loss：1.8964
第25个epoch，loss：1.8767
第26个epoch，loss：1.8564
第27个epoch，loss：1.8316
第28个epoch，loss：1.8086
第29个epoch，loss：1.7883
第30个epoch，loss：1.7683
第31个epoch，loss：1.7466
第32个epoch，loss：1.7254
第33个epoch，loss：1.7058
第34个epoch，loss：1.6855
第35个epoch，loss：1.6652
第36个epoch，loss：1.6498
第37个epoch，loss：1.6294
第38个epoch，loss：1.6099
第39个epoch，loss：1.5936
第40个epoch，loss：1.5728
第41个epoch，loss：1.5537
第42个epoch，loss：1.5370
第43个epoch，loss：1.5230
第44个epoch，loss：1.5000
第45个epoch，loss：1.4842
第46个epoch，loss：1.46

In [43]:
def predict_seq2seq(
    text,
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device
):
    net.eval()

    # 1. 清洗并分词英文
    text = preprocess_nmt(text)
    tokens = text.split()

    src_token_to_idx = src_vocab["token_to_idx"]
    tgt_token_to_idx = tgt_vocab["token_to_idx"]

    src_unk_idx = src_token_to_idx["<unk>"]
    src_eos_idx = src_token_to_idx["<eos>"]
    src_pad_idx = src_token_to_idx["<pad>"]

    bos_idx = tgt_token_to_idx["<bos>"]
    eos_idx = tgt_token_to_idx["<eos>"]

    # 2. 英文转编号
    x = [
        src_token_to_idx.get(
            token,
            src_unk_idx
        )
        for token in tokens
    ]

    x.append(src_eos_idx)

    x = truncate_pad(
        x,
        num_steps,
        src_pad_idx
    )

    x = torch.tensor(
        x,
        dtype=torch.long,
        device=device
    ).reshape(1, -1)
    # (1, num_steps)

    output_idx = []

    with torch.no_grad():
        # 3. Encoder
        encoder_state = net.encoder(x)

        forward_state = encoder_state[0]
        backward_state = encoder_state[1]

        context = torch.cat(
            (forward_state, backward_state),
            dim=1
        )

        context = torch.tanh(
            net.linear(context)
        )

        decoder_state = context.unsqueeze(0).repeat(
            net.decoder.gru.num_layers,
            1,
            1
        )

        # 4. Decoder第一个输入是<bos>
        decoder_x = torch.tensor(
            [[bos_idx]],
            dtype=torch.long,
            device=device
        )
        new_info=context.unsqueeze(0)

        # 5. 自回归生成
        for _ in range(num_steps):
            logit, decoder_state = net.decoder(
                decoder_x,
                decoder_state,
                new_info
            )

            predict_idx = logit.argmax(
                dim=1
            ).item()

            if predict_idx == eos_idx:
                break

            output_idx.append(predict_idx)

            # 当前预测作为下一步输入
            decoder_x = torch.tensor(
                [[predict_idx]],
                dtype=torch.long,
                device=device
            )

    # 6. 编号还原为法文
    result = [
        tgt_vocab["idx_to_token"][idx]
        for idx in output_idx
    ]

    return " ".join(result)

In [44]:
predict_seq2seq(
    "go",
    net,
    src_vocab,
    tgt_vocab,
    num_steps,
    device
)

'va !'